In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

## Load Dataset

In [10]:
df = pd.read_csv("../data/raw/netflix_titles.csv")
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## Basic Inspection

In [11]:
df.info()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


np.int64(0)

## Remove Duplicates

In [12]:
df.drop_duplicates(inplace=True)

## Handle Missing Values

In [14]:
df['director'] = df['director'].fillna("Unknown")
df['cast'] = df['cast'].fillna("Not Available")
df['country'] = df['country'].fillna("Unknown")
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

## Convert Date Column

In [16]:
# Clean whitespace
df['date_added'] = df['date_added'].str.strip()

# Convert to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Extract features
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

In [17]:
df['date_added'].isnull().sum()

np.int64(10)

In [18]:
df[df['date_added'].isnull()][['date_added']]

,date_added
6066,NaT
6174,NaT
6795,NaT
6806,NaT
6901,NaT
7196,NaT
7254,NaT
7406,NaT
7847,NaT
8182,NaT


## Normalize Country (Primary Country)

In [19]:
df['primary_country'] = df['country'].apply(lambda x: x.split(',')[0].strip())

## Normalize Genres

In [20]:
df['listed_in'] = df['listed_in'].str.lower()
df['genre_count'] = df['listed_in'].apply(lambda x: len(x.split(',')))

## Extract Numeric Duration

In [21]:
df['duration_num'] = df['duration'].str.extract('(\d+)')
df['duration_num'] = df['duration_num'].astype(float)

## Content Length Category

In [22]:
def categorize_length(row):
    if row['type'] == 'Movie':
        if row['duration_num'] < 60:
            return "Short Movie"
        elif row['duration_num'] <= 120:
            return "Medium Movie"
        else:
            return "Long Movie"
    else:
        if row['duration_num'] == 1:
            return "Limited Series"
        else:
            return "Multi Season"

df['content_length_category'] = df.apply(categorize_length, axis=1)

## Release Decade

In [24]:
df['release_decade'] = (df['release_year'] // 10) * 10

## Netflix Original Detection (Basic Logic)

In [25]:
df['is_original'] = df['title'].str.contains("Netflix", case=False, na=False)

In [28]:
df.to_csv("../data/processed/netflix_cleaned_featured.csv", index=False)